# Mis primeros códigos 

primero antes que nada hay que verificar y asegurarse de que todas laas librerías que necesitamos estén instaladas

In [ ]:
!pip install qiskit
!pip install jupyter
!pip install sympy
!pip install matplotlib
!pip install pylatexenc
!pip install numpy

una ves instaladas las librerias debemod llamarlas, el código abajo nos permitirá ver los datos del circuito sobre su representación en la esfera de bloch, el gráfico del circuito, el vector del circuito y el gráfico de probabilidades

In [ ]:
import os
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_bloch_multivector, array_to_latex, plot_histogram # ¡Agregamos plot_histogram!
from IPython.display import display, Markdown
from numpy import sqrt
def analizar_circuito(qc: QuantumCircuit, titulo: str = "Análisis del Circuito", guardar_img: str = None):
    """
    Muestra el circuito, su vector de estado (Ket), la esfera de Bloch,
    la matriz unitaria y el histograma de probabilidades en Jupyter Notebook.
    """
    display(Markdown(f"<br>## {titulo}"))
    display(Markdown("---"))
    
    # 1. Dibujo del circuito
    display(Markdown("**1. Diagrama del Circuito:**"))
    fig_circuito = qc.draw(output='mpl')
    display(fig_circuito)
    
    if guardar_img:
        os.makedirs("imagenes", exist_ok=True) 
        ruta_completa = os.path.join("imagenes", guardar_img)
        fig_circuito.savefig(ruta_completa)
        display(Markdown(f"*Circuito guardado en: `{ruta_completa}`*"))
    
    # Extraemos el vector de estado una sola vez para usarlo en los siguientes pasos
    estado = Statevector(qc)
    
    # 2. Vector de estado en notación Ket
    display(Markdown("<br>**2. Vector de Estado (Notación Ket):**"))
    display(estado.draw('latex'))
    
    # 3. Esferas de Bloch
    display(Markdown("<br>**3. Visualización en la Esfera de Bloch:**"))
    display(plot_bloch_multivector(estado))

    # 5. Histograma de Probabilidades (¡Lo nuevo!)
    display(Markdown("<br>**5. Probabilidades de Medición teóricas:**"))
    probabilidades = estado.probabilities_dict()
    # plot_histogram genera un gráfico de barras muy profesional
    grafico_probs = plot_histogram(probabilidades, title="Probabilidades de los Estados Base")
    display(grafico_probs)

# Ejercicios de producto tensorial
En esta sección se inicializan los vectores de estado canónicos del espacio de Hilbert de un qubit $\mathcal{H}_2$ mediante la clase `Statevector`, correspondientes a la base computacional $\{|0\rangle, |1\rangle\}$ y a las bases superpuestas:
* $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$
* $|-i\rangle = \frac{1}{\sqrt{2}}(|0\rangle - i|1\rangle)$ (denotado por la etiqueta `"l"` en Qiskit)

Asimismo, se definen los operadores elementales $\hat{O} \in \mathcal{L}(\mathcal{H}_2)$ utilizando `Operator`: la compuerta Hadamard ($H$), el operador identidad ($I$) y la compuerta Pauli-X ($X$).

In [ ]:
zero = Statevector.from_label("0")
one = Statevector.from_label("1")
plus = Statevector.from_label("+")
minus_i = Statevector.from_label("l")
H = Operator.from_label("H")
Id = Operator.from_label("I")
X = Operator.from_label("X")



### 1. Producto tensorial de estados idénticos

Se construye el estado compuesto $|\psi\rangle \in \mathcal{H}_2 \otimes \mathcal{H}_2$ mediante el producto tensorial de dos estados fundamentales $|1\rangle$:

$$|\psi\rangle = |1\rangle \otimes |1\rangle = |11\rangle$$

En Qiskit, esta operación se evalúa tanto con el operador sobrecargado `^` como con el método explícito `.tensor()`.

In [ ]:
display((one^one).draw('latex'))
psi = one.tensor(one)
display(psi.draw('latex'))

### Transformación unitaria conjunta: $H \otimes I$

En esta celda se calcula y visualiza la evolución del estado de dos qubits $|\phi\rangle$ bajo un operador compuesto:

1. **Producto tensorial de operadores (`H ^ Id`):** Se construye el operador global $U = H \otimes I$, donde la compuerta Hadamard ($H$) actúa sobre el primer subsistema y la identidad ($I$) sobre el segundo.
2. **Evolución del estado (`.evolve(...)`):** Se aplica la transformación lineal unitaria sobre el estado $|\phi\rangle = |+\rangle \otimes |-i\rangle$:

$$|\phi'\rangle = (H \otimes I)(|+\rangle \otimes |-i\rangle) = (H|+\rangle) \otimes (I|-i\rangle) = |0\rangle \otimes |-i\rangle = \frac{1}{\sqrt{2}}|00\rangle - \frac{i}{\sqrt{2}}|01\rangle$$

3. **Visualización (`.draw("latex")` y `display`):** Se renderiza el vector de estado resultante en notación matemática formal directamente en el notebook.

In [47]:
display(phi.evolve(H ^ Id).draw("latex"))

<IPython.core.display.Latex object>

### 4. Definición matricial de la compuerta CNOT y evolución

Se define de manera explícita el operador de dos qubits CNOT ($\text{CX}$) a través de su representación matricial estándar en la base computacional:

$$CX = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

Posteriormente, se hace evolucionar el estado separable $|\psi\rangle = |+\rangle \otimes |0\rangle$ bajo este operador para evaluar el estado entrelazado resultante (estado de Bell $|\Phi^+\rangle$).

In [46]:
CX = Operator([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0,0,1,0]])
psi = plus.tensor(zero)
display(psi.evolve(CX).draw("latex"))


<IPython.core.display.Latex object>

### Construcción y verificación de la compuerta CNOT ($CX$)

En este bloque se define el operador de dos qubits CNOT mediante su matriz explícita en la base computacional:

$$CX = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

Posteriormente, se evoluciona el estado separable $\vert{}\psi\rangle = \vert{}+\rangle \otimes \vert{}0\rangle$ bajo esta compuerta para generar el estado entrelazado de Bell $\vert{}\Phi^+\rangle = \frac{1}{\sqrt{2}}(\vert{}00\rangle + \vert{}11\rangle)$.

---

**Verificación de equivalencia con `QuantumCircuit`:**

Podemos comprobar que nuestra matriz coincide con la compuerta nativa `cx` de Qiskit extrayendo la matriz unitaria de un circuito mediante la clase `Operator`:

```python
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

# Circuito con compuerta CX nativa (control: qubit 0, target: qubit 1)
qc = QuantumCircuit(2)
qc.cx(1,0)

# Obtener operador unitario del circuito y comparar matrices
u = Operator(qc)
display(u.data)

In [48]:
qc = QuantumCircuit(2)
qc.cx(1,0)
u = Operator(qc)
display(u.data)

array([[1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
       [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j]])